In [2]:
%pip install transformers datasets accelerate evaluate

  Using cached transformers-5.8.1-py3-none-any.whl.metadata (33 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached huggingface_hub-1.14.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached multiprocess-0.70.19-py313-none-any.whl.metadata (7.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
Using cached transformers-5.8.1-py3-none-any.whl (10.6 MB)
Using cached huggingface_hub-1.14.0-py3-none-any.whl (661 kB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached datasets-4.8.5-py3-none-any.whl (528 kB)
Using cached multiprocess-0.70.19-py313-none-any.whl (156 kB)
Using cached accelera

In [3]:
# IMPORT LIBRARIES

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report
)

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [4]:
# LOAD CLEANED DATA

TRAIN_PATH = "../data/interim/train_clean_v2.csv"

train_df = pd.read_csv(TRAIN_PATH)

print(train_df.shape)

display(train_df.head())

(5000, 2)


,full_text,label
0,pret. di sekolah saya dapat mbg tetap saja ...,Sasaran Penerima
1,mbg bentuk penggarongan duit negara secara tsm...,Politik
2,pasal 34 ayat (1) undang-undang dasar negara r...,Sasaran Penerima
3,makan bergizi gratis bikin masyarakat merasa ...,Sasaran Penerima
4,"presiden ngotot, paling kesal kalau mbg dikri...",Politik


In [5]:
# LABEL ENCODING

labels = sorted(train_df['label'].unique())

label2id = {
    label: idx
    for idx, label in enumerate(labels)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

train_df['label_id'] = train_df['label'].map(label2id)

print(label2id)

{'Anggaran': 0, 'Distribusi': 1, 'Ekonomi': 2, 'Kualitas Pangan': 3, 'Lainnya': 4, 'Politik': 5, 'Sasaran Penerima': 6, 'Tata Kelola': 7}


In [6]:
# LOAD TOKENIZER

MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded.


In [7]:
# TOKENIZATION FUNCTION

MAX_LENGTH = 256


def tokenize_function(texts):
    
    return tokenizer(
        
        texts.tolist(),
        
        padding='max_length',
        
        truncation=True,
        
        max_length=MAX_LENGTH
    )

In [8]:
# STRATIFIED KFOLD

skf = StratifiedKFold(
    
    n_splits=5,
    
    shuffle=True,
    
    random_state=42
)

print("StratifiedKFold initialized.")

StratifiedKFold initialized.


In [9]:
# CUSTOM DATASET

from torch.utils.data import Dataset


class TextDataset(Dataset):
    
    def __init__(
        self,
        encodings,
        labels
    ):
        
        self.encodings = encodings
        self.labels = labels
    
    
    def __getitem__(self, idx):
        
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        
        item['labels'] = torch.tensor(
            self.labels[idx]
        )
        
        return item
    
    
    def __len__(self):
        
        return len(self.labels)

In [10]:
# METRIC FUNCTION

import evaluate

accuracy_metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    
    logits, labels = eval_pred
    
    predictions = np.argmax(logits, axis=-1)
    
    
    # balanced accuracy
    balanced_acc = balanced_accuracy_score(
        labels,
        predictions
    )
    
    
    return {
        "balanced_accuracy": balanced_acc
    }

In [ ]:
# CROSS VALIDATION TRAINING

fold_scores = []

oof_predictions = np.zeros(len(train_df))

for fold, (train_idx, valid_idx) in enumerate(
    skf.split(
        train_df['full_text'],
        train_df['label_id']
    )
):
    
    print("=" * 60)
    print(f"FOLD {fold + 1}")
    print("=" * 60)
    
        # SPLIT DATA    
    train_fold = train_df.iloc[train_idx]
    valid_fold = train_df.iloc[valid_idx]
    
        # TOKENIZATION    
    train_encodings = tokenize_function(
        train_fold['full_text']
    )
    
    valid_encodings = tokenize_function(
        valid_fold['full_text']
    )
    
        # DATASET    
    train_dataset = TextDataset(
        train_encodings,
        train_fold['label_id'].tolist()
    )
    
    valid_dataset = TextDataset(
        valid_encodings,
        valid_fold['label_id'].tolist()
    )
    
        # MODEL    
    model = AutoModelForSequenceClassification.from_pretrained(
        
        MODEL_NAME,
        
        num_labels=len(label2id),
        
        id2label=id2label,
        
        label2id=label2id
    )
    
        # TRAINING ARGUMENTS    
    training_args = TrainingArguments(
        
        output_dir=f"./fold_{fold+1}",
        
        eval_strategy="epoch",
        
        save_strategy="epoch",
        
        logging_strategy="epoch",
        
        num_train_epochs=2,
        
        per_device_train_batch_size=8,
        
        per_device_eval_batch_size=8,
        
        learning_rate=2e-5,
        
        weight_decay=0.01,
        
        load_best_model_at_end=True,
        
        metric_for_best_model="balanced_accuracy",
        
        greater_is_better=True,
        
        report_to="none"
    )
    
        # TRAINER    
    trainer = Trainer(
        
        model=model,
        
        args=training_args,
        
        train_dataset=train_dataset,
        
        eval_dataset=valid_dataset,
        
        compute_metrics=compute_metrics
    )
    
        # TRAIN    
    trainer.train()
    
        # EVALUATION    
    predictions_output = trainer.predict(
        valid_dataset
    )
    
    predictions = np.argmax(
        predictions_output.predictions,
        axis=-1
    )
    
    
    # save OOF prediction
    oof_predictions[valid_idx] = predictions
    
    
    # score
    score = balanced_accuracy_score(
        valid_fold['label_id'],
        predictions
    )
    
    fold_scores.append(score)
    
    
    print(f"Balanced Accuracy : {score:.4f}")
    print()

FOLD 1


[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]